### Task 1

In [3]:
import pandas as pd
import requests

# https://jsonplaceholder.typicode.com
# https://httpbin.org

In [4]:
class UserActivityAnalyzer:
    def __init__(self):
        self.url = 'https://jsonplaceholder.typicode.com'
        self.users = []
        self.user = {}
        self.stats = []
    
    def fetch_all_users(self):
        url = f'{self.url}/users'
        response = requests.get(url)
        data = response.json()
        self.users = data
        return self.users

    def fetch_user(self, user_id):
        url = f'{self.url}/users'
        params = {'ID': user_id}
        response = requests.get(url, params)
        data = response.json()
        self.user = data[0]
        return self.user
        
    def fetch_user_todos(self, user_id):
        url = f'{self.url}/todos'
        params = {'userId': user_id}
        response = requests.get(url, params)
        data = response.json()
        return data

    def calculate_user_stats(self, user, todos):
        total = len(todos)
        completed = sum(1 for task in todos if task['completed'])
        rate = (completed / total * 100)

        if rate <= 20:
            stars = 1
        elif rate <= 40:
            stars = 2
        elif rate <= 60:
            stars = 3
        elif rate <= 80:
            stars = 4
        else:
            stars = 5

        # print(completed, rate, type(rate), stars)

        stat = {
            'user_id': user['id'],
            'name': user['name'],
            'total_tasks': total,
            'completed_tasks': completed,
            'completion_rate': rate,
            'productivity_stars': stars
        }

        # print(stat)
        
        return stat

    def analyze_all_users(self):
        users = self.fetch_all_users()
        self.stats = []
        for user in users:
            self.stats.append(self.calculate_user_stats(user, self.fetch_user_todos(user['id'])))
        # print(self.stats)
        
    def print_report(self):
        if not self.stats:
            print("No data")
            return
        print(f"{'id':<4} {'Name':<25} {'Total tasks':<12} {'Done':<10} {'% Done':<13} {'*'}")
        print("-" * 70)
        for stat in self.stats:
            print(f"{stat['user_id']:<4} {stat['name']:<25} {stat['total_tasks']:<12} "
                  f"{stat['completed_tasks']:<10} {stat['completion_rate']:<13.1f} {stat['productivity_stars']*'*'}")
    
    def find_most_productive(self):
        return max(self.stats, key=lambda s: s['completion_rate'])

    def find_least_productive(self):
        return min(self.stats, key=lambda s: s['completion_rate'])

    def create_motivation_task(self, user_id):
        url = f'{self.url}/todos'
        data = {
            "userId": user_id,
            "title": "Улучшить продуктивность!",
            "completed": False
        }
        response = requests.post(url, json=data)
        return response.json()
            
    def test(self):
        print(1)

# users = UserActivityAnalyzer()

In [5]:
foo = UserActivityAnalyzer()

foo.analyze_all_users()

foo.print_report()

most_productive = foo.find_most_productive()
print(f'\nCамый продуктивный: {most_productive}')

least_productive = foo.find_least_productive()
print(f'\nНаименее продуктивный: {least_productive}')

task = foo.create_motivation_task(least_productive['user_id'])
print(f'\nСоздана задача: {task}')

id   Name                      Total tasks  Done       % Done        *
----------------------------------------------------------------------
1    Leanne Graham             20           11         55.0          ***
2    Ervin Howell              20           8          40.0          **
3    Clementine Bauch          20           7          35.0          **
4    Patricia Lebsack          20           6          30.0          **
5    Chelsey Dietrich          20           12         60.0          ***
6    Mrs. Dennis Schulist      20           6          30.0          **
7    Kurtis Weissnat           20           9          45.0          ***
8    Nicholas Runolfsdottir V  20           11         55.0          ***
9    Glenna Reichert           20           8          40.0          **
10   Clementina DuBuque        20           12         60.0          ***

Cамый продуктивный: {'user_id': 5, 'name': 'Chelsey Dietrich', 'total_tasks': 20, 'completed_tasks': 12, 'completion_rate': 60.0, 'p

### Task 2

In [6]:
import requests
import json
import time

In [7]:
class HTTPInspector:
    def __init__(self):
        self.url = 'https://httpbin.org'
        self.results = []
        
    def test_custom_headers(self):
        url = f'{self.url}/headers'
        headers = {
            'User-Agent': 'HTTPInspector/1.0',
            'X-Custom-Header': 'TestValue',
            'Accept-Language': 'ru-RU'
        }
        r = requests.get(url, headers=headers)
        data = r.json()["headers"]
        return (
            data.get('User-Agent') == 'HTTPInspector/1.0' and
            data.get('X-Custom-Header') == 'TestValue' and
            data.get('Accept-Language') == 'ru-RU'
        )

    def test_query_parameters(self):
        url = f"{self.url}/get"
        params = {
            'search': 'python',
            'limit': '10',
            'sort': 'desc'
        }
        r = requests.get(url, params=params)
        args = r.json()['args']
        return args == params
        
    def test_post_json(self) -> bool:
        url = f"{self.url}/post"
        json = {
            'username': 'student',
            'email': 'student@example.com',
            'skills': ['Python', 'HTTP', 'REST']
        }
        r = requests.post(url, json=json)
        data = r.json()['json']
        return data == json
    
    def test_post_form(self) -> bool:
        url = f"{self.url}/post"
        form = {
            'name': 'Ivan',
            'age': '25',
            'city': 'Moscow'
        }
        r = requests.post(url, data=form)
        print(type(r))
        data = r.json()['form']
        return data == form

    def test_status_codes(self):
        codes = [200, 404, 500, 301, 401]
        result = {}
        for code in codes:
            url = f'{self.url}/status/{code}'
            try:
                r = requests.get(url, allow_redirects=False)
                result[code] = (r.status_code == code)
                # r<Tab>
            except requests.RequestException:
                result[code] = False
        return result

    def test_timeout(self) -> Tuple[bool, Optional[str]]:
        url = f'{self.url}/delay/5'
        try:
            requests.get(url, timeout=3)
            return False
        except requests.exceptions.Timeout:
            return True

    def test_basic_auth(self):
        url = f'{self.url}/basic-auth/user/passwd'
        try:
            r = requests.get(url, auth=('user', 'passwd'))
            return (r.status_code == 200)
        except requests.RequestException:
            return False

    def test_cookies(self):
        with requests.Session() as session:
            session.get(f"{self.url}/cookies/set?name=value")
            
            r = session.get(f"{self.url}/cookies")
            cookies = r.json().get('cookies', {})
            if cookies.get('name') != 'value':
                return False

            session.get(f"{self.url}/cookies/delete?name")
            r = session.get(f"{self.url}/cookies")
            cookies = r.json().get('cookies', {})
            return not ("name" in cookies)
        


In [8]:
# test = HTTPInspector()
# results = {}
# print(f"{'':<30} {'Результаты тестов':<30}")

# print(f'1. {test.test_custom_headers()}')

# print(f'2. {test.test_query_parameters()}')

# print(f'3. {test.test_post_json()}')

# print(f'4. {test.test_post_form()}')

# print(f'5. {test.test_status_codes()}')

# print(f'6. {test.test_timeout()}')

# print(f'7. {test.test_basic_auth()}')

# print(f'8. {test.test_cookies()}')

In [9]:
test = HTTPInspector()
tests = [
    test.test_custom_headers,
    test.test_query_parameters,
    test.test_post_json,
    test.test_post_form,
    lambda: all(test.test_status_codes().values()),
    test.test_timeout,
    test.test_basic_auth,
    test.test_cookies
]

total = len(tests)
passed = 0

print(f"{'':<30} {'Результаты тестов':<30}")
for i, t in enumerate(tests, start=1):
    r = t()
    print(f"{i}. {r}")
    if r:
        passed += 1

perc = passed / total * 100
print(f'\nИтого: {passed}/{total} ({perc:.2f}%)')

                               Результаты тестов             
1. True
2. True
3. True
<class 'requests.models.Response'>
4. True
5. True
6. True
7. True
8. True

Итого: 8/8 (100.00%)


In [156]:
requests.models.Response?

Init signature: requests.models.Response()
Docstring:     
The :class:`Response <Response>` object, which contains a
server's response to an HTTP request.
File:           f:\python\lib\site-packages\requests\models.py
Type:           type
Subclasses:     

In [162]:
requests.request?

Signature: requests.request(method, url, **kwargs)
Docstring:
Constructs and sends a :class:`Request <Request>`.

:param method: method for the new :class:`Request` object: ``GET``, ``OPTIONS``, ``HEAD``, ``POST``, ``PUT``, ``PATCH``, or ``DELETE``.
:param url: URL for the new :class:`Request` object.
:param params: (optional) Dictionary, list of tuples or bytes to send
    in the query string for the :class:`Request`.
:param data: (optional) Dictionary, list of tuples, bytes, or file-like
    object to send in the body of the :class:`Request`.
:param json: (optional) A JSON serializable Python object to send in the body of the :class:`Request`.
:param headers: (optional) Dictionary of HTTP Headers to send with the :class:`Request`.
:param cookies: (optional) Dict or CookieJar object to send with the :class:`Request`.
:param files: (optional) Dictionary of ``'name': file-like-objects`` (or ``{'name': file-tuple}``) for multipart encoding upload.
    ``file-tuple`` can be a 2-tuple ``('

In [112]:
# users.calculate_user_stats(users.fetch_user(1), users.fetch_user_todos(1))

users.analyze_all_users()

users.print_report()

users.find_most_productive()
users.find_least_productive()
users.create_motivation_task(2)

[{'user_id': 1, 'name': 'Leanne Graham', 'total_tasks': 200, 'completed_tasks': 90, 'completion_rate': 45.0, 'productivity_stars': 3}, {'user_id': 2, 'name': 'Ervin Howell', 'total_tasks': 200, 'completed_tasks': 90, 'completion_rate': 45.0, 'productivity_stars': 3}, {'user_id': 3, 'name': 'Clementine Bauch', 'total_tasks': 200, 'completed_tasks': 90, 'completion_rate': 45.0, 'productivity_stars': 3}, {'user_id': 4, 'name': 'Patricia Lebsack', 'total_tasks': 200, 'completed_tasks': 90, 'completion_rate': 45.0, 'productivity_stars': 3}, {'user_id': 5, 'name': 'Chelsey Dietrich', 'total_tasks': 200, 'completed_tasks': 90, 'completion_rate': 45.0, 'productivity_stars': 3}, {'user_id': 6, 'name': 'Mrs. Dennis Schulist', 'total_tasks': 200, 'completed_tasks': 90, 'completion_rate': 45.0, 'productivity_stars': 3}, {'user_id': 7, 'name': 'Kurtis Weissnat', 'total_tasks': 200, 'completed_tasks': 90, 'completion_rate': 45.0, 'productivity_stars': 3}, {'user_id': 8, 'name': 'Nicholas Runolfsdott

NameError: name 'false' is not defined

In [ ]:
users.calculate_user_stats(users.fetch_all_users(1), users.fetch_user_todos(1))